# LangGraph: Agents as State Graphs

Companion notebook for the [LangGraph wiki page](https://ml-viz.vercel.app/wiki/langgraph).

To understand LangGraph's abstractions without depending on the library or any LLM API, we build a
**tiny LangGraph-style engine from scratch** in pure Python — `StateGraph` with typed state,
reducers, nodes, normal edges, and conditional edges — then express a **ReAct agent** as a compiled
graph driven by a deterministic mock LLM and a calculator tool. No network, no API keys.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Any

START, END = "__start__", "__end__"

## 1 — A minimal StateGraph engine

The whole framework, in ~40 lines. **State** is a dict; each key has a **reducer** describing how a
node's output is merged. **Nodes** are `state -> partial update`. **Edges** are either fixed
(`add_edge`) or **conditional** (`add_conditional_edges`, which calls a router to pick the next
node). `invoke` runs super-steps until control reaches `END`.

In [ ]:
def overwrite(old, new):
    return new

def append(old, new):
    return (old or []) + (new if isinstance(new, list) else [new])

class StateGraph:
    def __init__(self, reducers):
        self.reducers = reducers          # {field: reducer_fn}
        self.nodes = {}                   # name -> fn
        self.edges = {}                   # name -> next node (fixed)
        self.cond = {}                    # name -> (router_fn, {key: node})
        self.entry = None

    def add_node(self, name, fn):       self.nodes[name] = fn
    def add_edge(self, src, dst):
        if src == START: self.entry = dst
        else:            self.edges[src] = dst
    def add_conditional_edges(self, src, router, mapping):
        self.cond[src] = (router, mapping)

    def _merge(self, state, update):
        for k, v in update.items():
            reducer = self.reducers.get(k, overwrite)
            state[k] = reducer(state.get(k), v)
        return state

    def compile(self):
        return CompiledGraph(self)

class CompiledGraph:
    def __init__(self, g): self.g = g
    def invoke(self, state, max_supersteps=50, trace=False):
        g, node = self.g, self.g.entry
        for _ in range(max_supersteps):
            if node == END:
                return state
            if trace: print(f"  ▶ node: {node}")
            state = g._merge(state, g.nodes[node](state))   # run node, apply reducers
            if node in g.cond:                               # conditional edge
                router, mapping = g.cond[node]
                node = mapping[router(state)]
            else:                                            # fixed edge
                node = g.edges.get(node, END)
        raise RuntimeError("max super-steps exceeded — missing termination?")

## 2 — A mock LLM and a tool

To stay offline and deterministic, the 'LLM' is a rule-based stand-in: it scans the running
transcript and either emits a **tool call** or a **final answer**. This mirrors what a real model
returns (text vs. a structured function call) without any network.

In [ ]:
@dataclass
class Msg:
    role: str                     # 'user' | 'ai' | 'tool'
    text: str = ""
    tool_call: Any = None         # (name, args) or None

def calculator(expr):
    return eval(expr, {"__builtins__": {}}, {})   # sandboxed arithmetic

TOOLS = {"calculator": calculator}

def mock_llm(messages):
    """Deterministic policy: if the last message is a tool result, answer; else request the calc."""
    last = messages[-1]
    if last.role == "tool":
        return Msg(role="ai", text=f"The answer is {last.text}.")
    # first turn: pull the arithmetic expression out of the user question
    expr = messages[0].text.split(":", 1)[1].strip()
    return Msg(role="ai", tool_call=("calculator", {"expr": expr}))

## 3 — Wire up the ReAct graph

Two nodes (`agent`, `tools`), one conditional edge that routes to `tools` while the LLM keeps
requesting calls and to `END` once it answers, and the `tools → agent` **cycle**. Note the bounded
loop: `route` forces `END` after 8 steps.

In [ ]:
def agent_node(state):
    reply = mock_llm(state["messages"])
    return {"messages": [reply], "steps": state["steps"] + 1}

def tool_node(state):
    name, args = state["messages"][-1].tool_call
    result = TOOLS[name](**args)
    return {"messages": [Msg(role="tool", text=str(result))]}

def route(state):
    last = state["messages"][-1]
    if state["steps"] >= 8:                 # bounded loop guard
        return "end"
    return "tools" if last.tool_call is not None else "end"

g = StateGraph(reducers={"messages": append, "steps": overwrite})
g.add_node("agent", agent_node)
g.add_node("tools", tool_node)
g.add_edge(START, "agent")
g.add_conditional_edges("agent", route, {"tools": "tools", "end": END})
g.add_edge("tools", "agent")              # the cycle a plain chain can't express
app = g.compile()

init = {"messages": [Msg(role="user", text="compute: (2100000 / 1000) + 50")], "steps": 0}
final = app.invoke(init, trace=True)

print("\nTranscript:")
for m in final["messages"]:
    tag = f"[{m.role}]"
    print(f"  {tag:8} {m.text or f'tool_call={m.tool_call}'}")
print(f"\nsteps taken: {final['steps']}")

## 4 — Why the reducer matters

Swap the `messages` reducer from `append` to `overwrite` and the agent loses its transcript — each
node clobbers the history, exactly the pitfall called out on the wiki page.

In [ ]:
g_bad = StateGraph(reducers={"messages": overwrite, "steps": overwrite})  # WRONG reducer
for name, fn in [("agent", agent_node), ("tools", tool_node)]:
    g_bad.add_node(name, fn)
g_bad.add_edge(START, "agent")
g_bad.add_conditional_edges("agent", route, {"tools": "tools", "end": END})
g_bad.add_edge("tools", "agent")

bad = g_bad.compile().invoke({"messages": [Msg(role="user", text="compute: 10 + 5")], "steps": 0})
print(f"with overwrite reducer, transcript length = {len(bad['messages'])} (history was clobbered)")
print(f"with append reducer,    transcript length = {len(final['messages'])} (history preserved)")

## ✏️ Your turn

**Exercise.** Real agents need a **bounded loop** so a misbehaving router can't spin forever.
Build a graph with a single node `loop` whose router always tries to route back to `loop`, but cap
it: implement `make_bounded_router(limit)` returning a `route(state)` that returns `"again"` while
`state["steps"] < limit` and `"stop"` once the cap is hit. The `loop` node increments `steps`.

The compiled graph should terminate after exactly `limit` visits to `loop`.

In [ ]:
def loop_node(state):
    return {"steps": state["steps"] + 1}

def make_bounded_router(limit):
    def route(state):
        # TODO(you): return 'again' while under the limit, else 'stop'
        ...
    return route

def build_bounded_graph(limit):
    g = StateGraph(reducers={"steps": overwrite})
    g.add_node("loop", loop_node)
    g.add_edge(START, "loop")
    # TODO(you): add a conditional edge from 'loop' using make_bounded_router(limit),
    #            mapping 'again' -> 'loop' and 'stop' -> END
    ...
    return g.compile()

In [ ]:
# This assert cell passes silently when your implementation is correct.
for limit in (1, 3, 8):
    out = build_bounded_graph(limit).invoke({"steps": 0})
    assert out["steps"] == limit, f"expected {limit} loops, got {out['steps']}"
print("✓ bounded loop terminates at the cap for every limit")

<details>
<summary>Solution</summary>

```python
def make_bounded_router(limit):
    def route(state):
        return "again" if state["steps"] < limit else "stop"
    return route

def build_bounded_graph(limit):
    g = StateGraph(reducers={"steps": overwrite})
    g.add_node("loop", loop_node)
    g.add_edge(START, "loop")
    g.add_conditional_edges("loop", make_bounded_router(limit),
                            {"again": "loop", "stop": END})
    return g.compile()
```

The router is a pure function of state, the cap lives in `state["steps"]`, and the conditional edge
is the only thing standing between you and an infinite, runaway-cost loop — which is why every
production agent graph has one.

</details>